In [77]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Load a model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model = torch.load("model_shape_prediction.pth")

# Load a data as input (phases) to get shapes
phase_input = np.load('phases.npy')

/tmp/ipykernel_10670/137047732.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  loaded_model = torch.load("model_shape_prediction.pth")


In [78]:
type(phase_input)

numpy.ndarray

In [79]:
print(phase_input[:10])

[-5.23598776 -5.68427285 -6.12587861 -0.2800347  -0.71525035 -1.15019534
 -1.58643455 -2.02523595 -2.46864226 -2.91940786]


In [80]:
# Exchange numpy data for torch
phase_input = torch.from_numpy(phase_input).type(torch.float)

### Fine! We have prepared everything for filling in
1. Now, we will calculate shapes according vectors off phase dalay
2. For that let's turn on eval mode for prediction model and then will get shapes as output wise-element.
3. Every output will be saved into new vectors - vector of shape coefficients.

In [81]:
print(phase_input.size())
print(phase_input[:10])
print(phase_input.ndim)

torch.Size([113])
tensor([-5.2360, -5.6843, -6.1259, -0.2800, -0.7153, -1.1502, -1.5864, -2.0252,
        -2.4686, -2.9194])
1


In [118]:
# Определяем размеры
num_samples = phase_input.size(0)  # Количество строк (итераций)
output_dim = loaded_model(phase_input[0].unsqueeze(0)).size(0)  # Размер предсказания (второй индекс)

In [119]:
output_dim

2

In [121]:
# Создаем пустую матрицу для хранения результатов
output = torch.zeros((num_samples, output_dim), device=device)

In [128]:
output[:10]

tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]], device='cuda:0')

In [129]:
# Устанавливаем режим оценки
phase_input = phase_input.to(device)
loaded_model.to(device)
loaded_model.eval()

# Получаем предсказания и заполняем матрицу
with torch.no_grad():
    for i in range(num_samples):
        print(f"Value of phase_input {i} is {phase_input[i]}")
        prediction = loaded_model(phase_input[i].unsqueeze(0))
        print(f"Prediction for {i} is {prediction}")
        output[i] = prediction.squeeze(0)  # Сохраняем строку предсказания

# Проверка выходных данных
print("Выходные данные модели:", output[:10])


Value of phase_input 0 is -5.235987663269043
Prediction for 0 is tensor([-0.2716, -0.3418], device='cuda:0')
Value of phase_input 1 is -5.684272766113281
Prediction for 1 is tensor([-0.5911, -0.8116], device='cuda:0')
Value of phase_input 2 is -6.125878810882568
Prediction for 2 is tensor([-0.8674, -1.2180], device='cuda:0')
Value of phase_input 3 is -0.2800346910953522
Prediction for 3 is tensor([3.6208, 5.1913], device='cuda:0')
Value of phase_input 4 is -0.7152503728866577
Prediction for 4 is tensor([4.9599, 7.3049], device='cuda:0')
Value of phase_input 5 is -1.1501953601837158
Prediction for 5 is tensor([4.6756, 6.9241], device='cuda:0')
Value of phase_input 6 is -1.5864346027374268
Prediction for 6 is tensor([4.0443, 6.0037], device='cuda:0')
Value of phase_input 7 is -2.025235891342163
Prediction for 7 is tensor([3.3754, 5.0215], device='cuda:0')
Value of phase_input 8 is -2.468642234802246
Prediction for 8 is tensor([2.7274, 4.0688], device='cuda:0')
Value of phase_input 9 is -

Готово! Мы создали тензор, который описывает форму ячеек в заданной точке.
Теперь у нас есть форма и есть центры. Не хватает только нормали, и возможно там будет сложная математика...